Agent_Gradio



머신러닝 모델 기반 서비스인 경우 그라디오를 사용하면 쉽게 사용자 인터페이스를 만들 수 있다.

그라디오 챗봇과 OpenAI API연동

In [1]:
# =========================================================
# 패키지 설치
# =========================================================
# 역할:
# - Gradio : 웹 UI 생성
# - OpenAI : OpenAI API 사용
# =========================================================

!pip install -qU gradio   # 세션 재시작
!pip install -qU openai langchain-openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 23.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.9/121.9 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 558.5/558.5 kB 33.0 MB/s eta 0:00:00


In [2]:
from langchain_openai import ChatOpenAI
from google.colab import userdata
import os

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

response = llm.invoke("안녕? 넌 누구니?")

print(response.content)

안녕하세요! 저는 AI 언어 모델입니다. 여러분의 질문에 답하고, 정보를 제공하며, 다양한 주제에 대해 대화할 수 있습니다. 무엇을 도와드릴까요?


In [3]:
# =========================================================
# 라이브러리 import
# =========================================================

import os
import gradio as gr
from openai import OpenAI

# =========================================================
# OpenAI Client 생성
# =========================================================

client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY")
)

In [ ]:
# =========================================================
# 역할:
# - 사용자 질문을 OpenAI에 전달
# - GPT 응답 생성
# - 채팅 기록 저장
# - Gradio 채팅창 출력
# =========================================================

# =========================================================
# 챗봇 응답 함수
# =========================================================

def respond(message, chat_history):

    # -----------------------------------------------------
    # chat_history가 None인 경우 초기화
    # -----------------------------------------------------

    if chat_history is None:

        # 빈 리스트 생성
        chat_history = []

    # -----------------------------------------------------
    # OpenAI API 호출
    # -----------------------------------------------------

    response = client.chat.completions.create(

        # 사용할 모델
        model= "gpt-4o-mini",

        # 사용자 메시지 전달
        messages= [
            {
                "role" : "user",
                "content" : message
            }
        ],

        # 최대 응답 길이
        max_tokens= 256,

        # 응답 랜덤성 제거
        temperature= 0
    )

    # -----------------------------------------------------
    # GPT 응답 추출
    # -----------------------------------------------------

    bot_message = response.choices[0].message.content

    # -----------------------------------------------------
    # 사용자 메시지 저장
    # -----------------------------------------------------

    chat_history.append(
        {
            "role" : "user",
            "content" : message
        }
    )

    # -----------------------------------------------------
    # GPT 응답 저장
    # -----------------------------------------------------

    chat_history.append(
        {
            "role" : "assistant",
            "content" : bot_message
        }
    )

    # -----------------------------------------------------
    # 입력창 초기화 및 채팅 기록 반환
    # -----------------------------------------------------

    return "", chat_history


# =========================================================
# Gradio UI 생성
# =========================================================

with gr.Blocks() as demo:

    # -----------------------------------------------------
    # 채팅창 생성
    # -----------------------------------------------------

    chatbot = gr.Chatbot(

        # 채팅창 제목
        label="채팅창"
    )

    # -----------------------------------------------------
    # 입력창 생성
    # -----------------------------------------------------

    msg = gr.Textbox(

        # 입력창 제목
        label="입력"
    )

    # -----------------------------------------------------
    # 초기화 버튼 생성
    # -----------------------------------------------------

    clear = gr.Button(
        "초기화"
    )

    # -----------------------------------------------------
    # 엔터 입력 시 응답 생성
    # -----------------------------------------------------

    msg.submit(

        # 실행 함수
        fn= respond,

        # 입력 컴포넌트
        inputs=[msg, chatbot],

        # 출력 컴포넌트
        outputs=[msg, chatbot]
    )

    # -----------------------------------------------------
    # 채팅창 초기화
    # -----------------------------------------------------

    clear.click(

        # 빈 리스트 반환
        fn= lambda : [],

        # 출력 대상
        outputs= chatbot,

        # 큐 사용 안함
        queue= False
    )

# =========================================================
# Gradio 실행
# =========================================================

demo.launch(share=True)

# =========================================================
# 실행 흐름
#
# 사용자 입력
#      ↓
# respond()
#      ↓
# OpenAI API
#      ↓
# GPT 응답 생성
#      ↓
# MessageDict 저장
#      ↓
# Gradio Chatbot 출력
# =========================================================

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://265b5224169465df8d.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
# =========================================================
# 역할:
# - 사용자 질문을 OpenAI에 전달
# - GPT 응답 생성
# - Gradio 채팅창 출력
# - 채팅 기록 초기화
# - 대화 종료 기능 제공
# =========================================================

# =========================================================
# 챗봇 응답 함수
# =========================================================

def respond(message, chat_history):

    # -----------------------------------------------------
    # chat_history 초기화
    # -----------------------------------------------------

    if chat_history is None:
        chat_history = []

    # -----------------------------------------------------
    # OpenAI API 호출
    # -----------------------------------------------------

    response = client.chat.completions.create(
          model = "gpt-4o-mini",

          messages = [
              {
                  "role" : "user",
                  "content" : message
              }
          ],

          max_tokens= 256,
          temperature=0
    )

    # -----------------------------------------------------
    # GPT 응답 추출
    # -----------------------------------------------------

    bot_message = response.choices[0].message.content

    # -----------------------------------------------------
    # 사용자 메시지 저장
    # -----------------------------------------------------

    chat_history.append(
        {
            "role" : "user",
            "content" : message
        }
    )

    # -----------------------------------------------------
    # GPT 응답 저장
    # -----------------------------------------------------

    chat_history.append(
        {
            "role" : "assistant",
            "content" : bot_message
        }
    )

    # -----------------------------------------------------
    # 입력창 초기화
    # -----------------------------------------------------

    return "", chat_history


# =========================================================
# 대화 종료 함수
# =========================================================

def end_chat():

    # -----------------------------------------------------
    # 채팅 기록 삭제
    # -----------------------------------------------------

    chat_history = []

    # -----------------------------------------------------
    # 입력창 비활성화
    # -----------------------------------------------------

    textbox_update = gr.update(

        # 입력 금지
        interactive= False,

        # 안내 문구
        placeholder= "대화가 종료되었습니다."
    )

    # -----------------------------------------------------
    # 채팅창 + 입력창 상태 반환
    # -----------------------------------------------------

    return chat_history, textbox_update


# =========================================================
# Gradio UI 생성
# =========================================================

with gr.Blocks() as demo:

    # -----------------------------------------------------
    # 채팅창 생성
    # -----------------------------------------------------

    chatbot = gr.Chatbot(
        label="채팅창"
    )


    # -----------------------------------------------------
    # 입력창 생성
    # -----------------------------------------------------

    msg = gr.Textbox(
        label="입력"
    )


    # -----------------------------------------------------
    # 초기화 버튼 생성
    # -----------------------------------------------------

    clear = gr.Button(
        "초기화"
    )


    # -----------------------------------------------------
    # 대화 종료 버튼 생성
    # -----------------------------------------------------

    exit_btn = gr.Button(
        "대화 종료"
    )

    # -----------------------------------------------------
    # 엔터 입력 시 GPT 호출
    # -----------------------------------------------------

    msg.submit(

        fn= respond,

        inputs = [msg, chatbot],

        outputs = [msg, chatbot]

    )

    # -----------------------------------------------------
    # 채팅 기록 초기화
    # -----------------------------------------------------

    clear.click(

        fn= lambda: [],

        outputs = chatbot,

        queue = False
    )

    # -----------------------------------------------------
    # 대화 종료
    # -----------------------------------------------------

    exit_btn.click(

        # 실행 함수
        fn= end_chat,

        # 출력 대상
        outputs= [chatbot, msg],

        # 즉시 실행
        queue=False
    )

# =========================================================
# Gradio 실행
# =========================================================

demo.launch(

    # 디버그 모드
    debug=True
)

# =========================================================
# 실행 흐름
#
# 사용자 입력
#      ↓
# OpenAI API
#      ↓
# GPT 응답 생성
#      ↓
# Chatbot 출력
#
# [초기화]
#      ↓
# 채팅 기록 삭제
#
# [대화 종료]
#      ↓
# 채팅 기록 삭제
#      ↓
# 입력창 비활성화
#      ↓
# 추가 입력 불가
# =========================================================

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://26a172cb5e874e801a.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7863 <> https://26a172cb5e874e801a.gradio.live


PDF 파일 기반 질의 응답 챗봇
  - 랭체인
  - 그라디오
  - ChatGPT

In [ ]:
# =========================================================
# 역할:
# - LangChain 기반 PDF 질의응답 챗봇 개발 환경 구성
# - PDF 로딩
# - 문서 분할
# - 벡터DB 저장
# - 토큰 계산
# =========================================================

# ---------------------------------------------------------
# LangChain 핵심 패키지
# ---------------------------------------------------------

!pip install -qU langchain

# ---------------------------------------------------------
# LangChain Community
# - PDF Loader
# - Vector DB
# - 다양한 외부 연동
# ---------------------------------------------------------

!pip install -qU langchain-community

# ---------------------------------------------------------
# OpenAI 연동
# - ChatOpenAI
# - OpenAIEmbeddings
# ---------------------------------------------------------

!pip install -qU langchain-openai

# ---------------------------------------------------------
# PDF 문서 로딩
# ---------------------------------------------------------

!pip install -qU pypdf

# ---------------------------------------------------------
# Vector DB
# - ChromaDB 사용
# ---------------------------------------------------------

!pip install -qU chromadb

# ---------------------------------------------------------
# 토큰 계산
# ---------------------------------------------------------

!pip install -qU tiktoken

# ---------------------------------------------------------
# Gradio UI
# ---------------------------------------------------------

!pip install -qU gradio

# ---------------------------------------------------------
# FAISS 설치
# - 벡터 DB 생성용
# ---------------------------------------------------------

!pip install -qU faiss-cpu

# =========================================================
# 설치 확인
# =========================================================

import langchain
import gradio

print("LangChain Version :", langchain.__version__)
print("Gradio Version    :", gradio.__version__)

# =========================================================
# 설치 목적
#
# PDF
#   ↓
# PyPDFLoader
#   ↓
# Text Splitter
#   ↓
# OpenAI Embeddings
#   ↓
# ChromaDB
#   ↓
# Retriever
#   ↓
# ChatOpenAI
#   ↓
# Gradio Chatbot
# =========================================================

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.6/139.6 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 41.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 27.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 17.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 65.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━

In [ ]:

# =========================================================
# PDF Loader
# =========================================================

# PDF 파일 로드
from langchain_community.document_loaders import PyPDFLoader

# =========================================================
# Embedding Model
# =========================================================

# OpenAI Embedding 모델
from langchain_openai import OpenAIEmbeddings

# =========================================================
# Text Splitter
# =========================================================

# 문서를 일정 길이로 분할
from langchain_text_splitters import RecursiveCharacterTextSplitter

# =========================================================
# Vector DB
# =========================================================

# Chroma Vector DB
from langchain_community.vectorstores import Chroma

# =========================================================
# (선택) FAISS Vector DB
# =========================================================

# 실무에서는 Chroma보다 FAISS를 더 자주 사용
from langchain_community.vectorstores import FAISS





In [ ]:
# =========================================================
# Colab Secret 사용
# =========================================================

import os
from google.colab import userdata

# Colab Secret에서 API Key 읽기
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")


In [ ]:
# 로컬 PC 업로드

# from google.colab import file

# uploaded = files.upload()

# filename = list(uploaded.key())[0]
# loader = PyPDFLoader(f'/content/{filename}')

In [ ]:
# =========================================================
# 역할:
# - PDF 문서 로드
# - 문서 분할(Chunking)
# - RAG 검색을 위한 전처리
# =========================================================

# =========================================================
# PDF 파일 로드
# =========================================================

# PDF URL
pdf_url = (
    "http://kocw-n.xcache.kinxcdn.com/data/keris/2023/keris0514/01.pdf"
)

# PDF Loader 생성
loader = PyPDFLoader(pdf_url)

# PDF 문서 로드
documents = loader.load()

# =========================================================
# 문서 분할기 생성
# =========================================================

# 최신 LangChain 권장 방식
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 200,
    length_function = len
)

# =========================================================
# 문서 분할
# =========================================================

texts = text_splitter.split_documents(
    documents
)

# =========================================================
# 결과 확인
# =========================================================

# 원본 페이지 수
print(
    f"PDF 페이지 수 : {len(documents)}"
)

# 생성된 Chunk 수
print(
    f"Chunk 수 : {len(texts)}"
)

# 첫 번째 Chunk 길이
print(
    f"첫 번째 Chunk 길이 : "
    f"{len(texts[0].page_content)}"
)

# =========================================================
# 첫 번째 Chunk 내용 확인
# =========================================================

print(
  texts[0].page_content[:500]

)

# =========================================================
# 실행 흐름
#
# PDF URL
#      ↓
# PyPDFLoader
#      ↓
# documents
#      ↓
# RecursiveCharacterTextSplitter
#      ↓
# texts(Chunk)
#      ↓
# Embedding
#      ↓
# Vector DB
# =========================================================

PDF 페이지 수 : 11
Chunk 수 : 26
첫 번째 Chunk 길이 : 890
인공지능으로 인한 변화와 윤리적 이슈1. 4차 산업혁명을 통한 인공지능의 등장과 사회적 변화[학습목표]Ÿ인공지능이 우리 산업에 미치는 영향을 사례를 들어 설명할 수 있다.Ÿ인공지능이 우리 직업에 미치는 영향을 사례를 들어 설명할 수 있다.[학습내용]Ÿ인공지능으로 인한 산업의 변화Ÿ인공지능으로 인한 직업의 변화(1) 산업의 변화가. 4차 산업혁명의 개념4차 산업혁명으로 인공지능(AI), 빅데이터와 같은 신기술이 우리 경제와 사회를 변화시키는 원동력이 되고 있다. 클라우스 슈밥(Klasu Schwab) 의장은 2016년 세계경제포럼(World Econmic Forum, WEF)에서 4차 산업혁명을 핵심 의제로 삼았다. 4차 산업혁명은 3차 산업혁명을 기반으로 한 디지털과 바이오산업, 물리학 등의 경계를 융합하는 기술 혁명을 말한다. 사물인터넷(IoT), 로봇(Robot), 인공지능(AI), 빅데이터(Bigdata) 등의 신기술과 나노기술(NT), 바이오기술(BT), 정보기술(IT), 인


In [ ]:
# =========================================================
# 역할:
# - 분할된 문서(Chunk) 확인
# - Chunk 개수 확인
# - Chunk 내용 확인
# =========================================================


# =========================================================
# 전체 Chunk 개수 확인
# =========================================================

print(
    f"생성된 Chunk 수 : {len(texts)}"
)

# =========================================================
# 첫 번째 Chunk 정보 확인
# =========================================================

print(
    texts[0]
)

# =========================================================
# 첫 번째 Chunk 내용 확인
# =========================================================

print(
    texts[0].page_content
)

# =========================================================
# 첫 번째 Chunk 메타데이터 확인
# =========================================================

print(
    texts[0].metadata
)

# =========================================================
# 실행 흐름
#
# PDF
#   ↓
# Splitter
#   ↓
# texts
#   ↓
# Chunk 확인
# =========================================================

생성된 Chunk 수 : 26
page_content='인공지능으로 인한 변화와 윤리적 이슈1. 4차 산업혁명을 통한 인공지능의 등장과 사회적 변화[학습목표]Ÿ인공지능이 우리 산업에 미치는 영향을 사례를 들어 설명할 수 있다.Ÿ인공지능이 우리 직업에 미치는 영향을 사례를 들어 설명할 수 있다.[학습내용]Ÿ인공지능으로 인한 산업의 변화Ÿ인공지능으로 인한 직업의 변화(1) 산업의 변화가. 4차 산업혁명의 개념4차 산업혁명으로 인공지능(AI), 빅데이터와 같은 신기술이 우리 경제와 사회를 변화시키는 원동력이 되고 있다. 클라우스 슈밥(Klasu Schwab) 의장은 2016년 세계경제포럼(World Econmic Forum, WEF)에서 4차 산업혁명을 핵심 의제로 삼았다. 4차 산업혁명은 3차 산업혁명을 기반으로 한 디지털과 바이오산업, 물리학 등의 경계를 융합하는 기술 혁명을 말한다. 사물인터넷(IoT), 로봇(Robot), 인공지능(AI), 빅데이터(Bigdata) 등의 신기술과 나노기술(NT), 바이오기술(BT), 정보기술(IT), 인지과학(CS) 등의 융합 기술이 생산을 주도하는 사회 구조적 혁명을 의미한다(이상동, 2016)지금까지의 산업혁명의 변천 과정을 살펴보면 <표 1>과 같다. 증기 기관을 활용한 1차 산업혁명은 사람이나 동물에 의한 동력이 기계로 전환되면서 생산량을 획기적으로 증가시켰다. 이후 2차 산업혁명은 전기에너지를 활용하여 좁은 공간에서도 대량 생산이 가능해졌으며, 3차 산업혁명은 ICT(Information and Communication Technology)를 활용한 자동화로 인해 로봇이 사람을 대신하게 되어 적은 비용으로 대량 생산이 가능하게 하였다. 4차 산업혁명은 인공지능(AI) 기술, 네트워크 전송 비용, 데이터 저장과 처리 비용, 분석 기술의 발달, 스마트 기기의 기술 혁신 등으로 도래하' metadata={'producer': 'Hancom PDF 1.3.0.538', 'creator': 'Hwp 2018 11.0.0.212

In [ ]:
# 분할된 텍스트 청크를 출력합니다.
texts

[Document(metadata={'producer': 'Hancom PDF 1.3.0.538', 'creator': 'Hwp 2018 11.0.0.2129', 'creationdate': '2023-05-09T16:10:07+09:00', 'author': 'cicsoft', 'moddate': '2023-05-09T16:10:07+09:00', 'pdfversion': '1.4', 'source': 'http://kocw-n.xcache.kinxcdn.com/data/keris/2023/keris0514/01.pdf', 'total_pages': 11, 'page': 0, 'page_label': '1'}, page_content='인공지능으로 인한 변화와 윤리적 이슈1. 4차 산업혁명을 통한 인공지능의 등장과 사회적 변화[학습목표]Ÿ인공지능이 우리 산업에 미치는 영향을 사례를 들어 설명할 수 있다.Ÿ인공지능이 우리 직업에 미치는 영향을 사례를 들어 설명할 수 있다.[학습내용]Ÿ인공지능으로 인한 산업의 변화Ÿ인공지능으로 인한 직업의 변화(1) 산업의 변화가. 4차 산업혁명의 개념4차 산업혁명으로 인공지능(AI), 빅데이터와 같은 신기술이 우리 경제와 사회를 변화시키는 원동력이 되고 있다. 클라우스 슈밥(Klasu Schwab) 의장은 2016년 세계경제포럼(World Econmic Forum, WEF)에서 4차 산업혁명을 핵심 의제로 삼았다. 4차 산업혁명은 3차 산업혁명을 기반으로 한 디지털과 바이오산업, 물리학 등의 경계를 융합하는 기술 혁명을 말한다. 사물인터넷(IoT), 로봇(Robot), 인공지능(AI), 빅데이터(Bigdata) 등의 신기술과 나노기술(NT), 바이오기술(BT), 정보기술(IT), 인지과학(CS) 등의 융합 기술이 생산을 주도하는 사회 구조적 혁명을 의미한다(이상동, 2016)지금까지의 산업혁명의 변천 과정을 살펴보면 <표 1>과 같다. 증기 기관을 활용한 1차 산업혁명은 사람이나 동물에 의한 동력이 기계로 전환되면서 생산량을 

In [ ]:
# =========================================================
# 역할:
# - 문서 Chunk 임베딩
# - FAISS Vector DB 생성
# - Retriever 생성
# - RAG 검색 준비
# =========================================================


# =========================================================
# 필요 Import
# =========================================================

# FAISS Vector DB
from langchain_community.vectorstores import FAISS

# =========================================================
# OpenAI Embedding 모델 생성
# =========================================================

embeddings = OpenAIEmbeddings(
    model = "text-embedding-3-small"
)

# =========================================================
# FAISS Vector DB 생성
# =========================================================

vector_store = FAISS.from_documents(
    documents= texts,
    embedding= embeddings
)

# =========================================================
# Retriever 생성
# =========================================================

retriever = vector_store.as_retriever(
    search_type = "similarity",
    search_kwargs= {
        "k": 2
    }
)

# =========================================================
# Retriever 테스트
# =========================================================

test_docs = retriever.invoke(
    "강의의 주요 내용은 무엇인가?"
)

# =========================================================
# 검색 결과 확인
# =========================================================

print(
    f"검색 문서 수 : {len(test_docs)}"
)

print("\n첫 번째 검색 문서\n")

print(
    test_docs[0].page_content[:500]
)

# =========================================================
# 실행 흐름
#
# texts
#    ↓
# OpenAIEmbeddings
#    ↓
# Vector 생성
#    ↓
# FAISS
#    ↓
# Retriever
#    ↓
# Similarity Search
# =========================================================

검색 문서 수 : 2

첫 번째 검색 문서

각기 다른 직무나 지식, 또는 기술과 기능을 합쳐 새로운 전문 분야를 창출하는 ‘블렌딩(blending)’ 과정을 통해 기존에는 이질적이었던 복수 직업 간 결합으로 새로운 융합형 직업이 출현할 것이다. 융합형 직업은 작게는 사람들이 가진 소질과 관심의 결합에서부터 기술 또는 지식 간, 그리고 과학기술과 타 영역 간 연결 과정에서 발생할 수 있다. 과학기술로 탄생하는 새로운 직업의 생성은 현재 우리가 상상하지 못하는 영역에서 창출될 가능성이 높다. 불과 10년 전에는 없었으나 지금 각광받는 직업을 살펴보면, 대부분의 직업들이 기술 진보로 인해 새로운 영역을 구축한 경우에 속한다. 예를 들어, 드론 조종사는 드론 기술이 상용화됨에 따라 현실의 직업으로 부상한 것이다. 
2. 인공지능(AI)으로 인한 윤리 문제[학습목표]Ÿ인공지능으로 발생되는 윤리적 이슈를 찾아 대안을 생각할 수 있다.Ÿ인공지능을 올바르게 사용하기 위한 방안이 무엇인지 설명할 수 있다.[학습내용]Ÿ인공지능의 윤리적 이슈Ÿ


In [ ]:
# =========================================================
# 역할:
# - ChatOpenAI 생성
# - Retriever 연결
# - LCEL RAG Chain 생성
# - LangChain 1.3.10 표준 적용
# =========================================================

# =========================================================
# 필요 Import
# =========================================================

# 최신 OpenAI Chat Model
from langchain_openai import ChatOpenAI

# Prompt Template
from langchain_core.prompts import ChatPromptTemplate

# Output Parser
from langchain_core.output_parsers import StrOutputParser

# Runnable 객체
from langchain_core.runnables import RunnablePassthrough

# =========================================================
# LLM 생성
# =========================================================

llm = ChatOpenAI(

    # 사용할 모델
    model="gpt-4o-mini",

    # 동일 입력 → 동일 출력
    temperature=0
)

# =========================================================
# Prompt 생성
# =========================================================

prompt = ChatPromptTemplate.from_template(
    """
    다음 문서를 기반으로 질문에 답변하시오.

    문서:
    {context}
    질문:
    {question}
    답변:
    """
)

# =========================================================
# 검색된 문서를 문자열로 변환
# =========================================================

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


# =========================================================
# LCEL RAG Chain 생성
# =========================================================

chain = (
    {
        "context" : retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt | llm | StrOutputParser()
)

# =========================================================
# 테스트 질문
# =========================================================

question = "강의의 주요 내용은 무엇인가?"

# =========================================================
# Chain 실행
# =========================================================

response = chain.invoke(
    question
)


# =========================================================
# 결과 출력
# =========================================================

print("\n==============================")
print("질문")
print("==============================")
print(question)

print("\n==============================")
print("답변")
print("==============================")
print(response)

# =========================================================
# 실행 흐름
#
# 질문
#    ↓
# Retriever
#    ↓
# 관련 문서 검색
#    ↓
# format_docs()
#    ↓
# Prompt 생성
#    ↓
# ChatOpenAI
#    ↓
# StrOutputParser
#    ↓
# 최종 답변 출력
# =========================================================


질문
강의의 주요 내용은 무엇인가?

답변
강의의 주요 내용은 '블렌딩(blending)' 과정을 통해 새로운 융합형 직업이 출현할 가능성과 인공지능(AI)으로 인한 윤리적 이슈에 대한 논의입니다. 블렌딩 과정은 다양한 직무, 지식, 기술을 결합하여 새로운 전문 분야를 창출하는 것을 의미하며, 이는 과학기술의 발전으로 인해 새로운 직업이 생성될 수 있음을 보여줍니다. 예를 들어, 드론 조종사와 같은 직업이 기술의 상용화로 인해 등장한 사례가 있습니다.

또한, 인공지능의 윤리적 이슈로는 차별과 편견이 있으며, 이는 편협한 데이터나 잘못된 알고리즘으로 인해 발생할 수 있습니다. 구글 포토의 사례와 COMPAS 시스템의 문제를 통해 이러한 이슈가 구체적으로 설명됩니다. 마지막으로, 감정 인식, 표현, 생성 단계로 구성된 사회 친화적인 로봇의 개발과 그로 인한 인간과의 정서적 교감에 대한 내용도 포함되어 있습니다.


In [ ]:
# =========================================================
# 역할:
# - 사용자 질문 정의
# - LCEL Chain 실행
# - 답변 출력
# =========================================================

# =========================================================
# 사용자 질문
# =========================================================

query = "자율자동차의 지원은?"

# =========================================================
# LCEL Chain 실행
# =========================================================

result = chain.invoke(

    # 사용자 질문
    query
)

# =========================================================
# 결과 출력
# =========================================================

print("\n==============================")
print("질문")
print("==============================")
print(query)

print("\n==============================")
print("답변")
print("==============================")
print(result)

# =========================================================
# 실행 흐름
#
# 사용자 질문
#      ↓
# chain.invoke()
#      ↓
# Retriever
#      ↓
# 관련 문서 검색
#      ↓
# Prompt 생성
#      ↓
# ChatOpenAI
#      ↓
# 답변 생성
#      ↓
# 결과 출력
# =========================================================


질문
자율자동차의 지원은?

답변
자율주행차의 지원은 주로 운전자의 간섭 없이 스스로 움직일 수 있도록 하는 기술적 기능을 포함합니다. 이러한 차량은 목적지까지 안전하게 이동할 수 있으며, 사람의 실수를 줄여 교통사고를 감소시키는 데 기여합니다. 그러나 자율주행차는 예측할 수 없는 상황에 직면할 수 있으며, 이때 윤리적인 선택이 필요할 수 있습니다. 예를 들어, 사고 상황에서 누구를 우선적으로 구해야 하는지에 대한 딜레마가 발생할 수 있습니다. 이러한 문제는 개인의 양심에 따라 결정되어야 하며, 개발자와 사용자 모두의 윤리가 중요합니다. 따라서 자율주행차의 설계와 운영에 있어서는 각 지역의 문화와 사회적 합의를 반영한 윤리 기준이 필요합니다.


In [ ]:
result

'자율주행차의 지원은 주로 운전자의 간섭 없이 스스로 움직일 수 있도록 하는 기술적 기능을 포함합니다. 이러한 차량은 목적지까지 안전하게 이동할 수 있으며, 사람의 실수를 줄여 교통사고를 감소시키는 데 기여합니다. 그러나 자율주행차는 예측할 수 없는 상황에 직면할 수 있으며, 이때 윤리적인 선택이 필요할 수 있습니다. 예를 들어, 사고 상황에서 누구를 우선적으로 구해야 하는지에 대한 딜레마가 발생할 수 있습니다. 이러한 문제는 개인의 양심에 따라 결정되어야 하며, 개발자와 사용자 모두의 윤리가 중요합니다. 따라서 자율주행차의 설계와 운영에 있어서는 각 지역의 문화와 사회적 합의를 반영한 윤리 기준이 필요합니다.'

In [ ]:
# =========================================================
# 역할:
# - Retriever가 찾은 문서 확인
# - 출처(Source) 확인
# =========================================================

# 검색 문서 조회
docs = retriever.invoke(query)

# 검색된 문서 수
print(f"검색 문서 수 : {len(docs)}")

# 첫 번째 문서 메타데이터 출력
print(docs[0].metadata)

# 첫 번째 문서 내용 일부 출력
print(docs[0].page_content[:500])

검색 문서 수 : 2
{'producer': 'Hancom PDF 1.3.0.538', 'creator': 'Hwp 2018 11.0.0.2129', 'creationdate': '2023-05-09T16:10:07+09:00', 'author': 'cicsoft', 'moddate': '2023-05-09T16:10:07+09:00', 'pdfversion': '1.4', 'source': 'http://kocw-n.xcache.kinxcdn.com/data/keris/2023/keris0514/01.pdf', 'total_pages': 11, 'page': 9, 'page_label': '10'}
있다. 왼쪽에는 SUV 차가, 오른쪽에는 소형 오토바이가 달리고 있다. 이때 대형 화물차의 짐이 떨어지고 있다. 그대로 직진하면 짐에 부딪혀 자율주행차의 운전자가 크게 다치게 된다. 왼쪽으로 이동하면  SUV 차와 부딪히게 되어 운전자가 다칠 수 있으며, 오른쪽으로 이동하면 오토바이 운전자는 크게 다치게 되지만 자율주행차의 운전자는 덜 다칠 수 있다. 사람들은 어떻게 설계된 자율주행차를 선택할까?
[그림]  자율주행차의 딜레마 Ÿ자율주행차는 운전자의 간섭 없이 스스로 움직이는 자동차이다. 운전자가 운전하지 않아도 목적지까지 안전하게 이동할 수 있으며, 사람의 실수를 줄여서 교통사고를 줄일 수 있다. 하지만, 아무리 뛰어난 인공지능(AI)이 탑재되어 있더라도 예측할 수 없는 상황이 발생할 수 있고, 트롤리 딜레마처럼 윤리적인 선택이 필요한 경우가 있다. 사고가 났을 때, 우선적으로 운전자를 살리는 차와 가능한 많은 사람을 살리는 차 중에서 어떤 것을 구매할 것인지의 선택은 운전자의 몫


In [ ]:
# =========================================================
# 역할:
# - Chat Prompt 관련 클래스 Import
# - LangChain 1.3.10 표준 적용
# =========================================================

# =========================================================
# 실행 흐름
#
# System Prompt
#        ↓
# Human Prompt
#        ↓
# ChatPromptTemplate
#        ↓
# ChatOpenAI
#        ↓
# 답변 생성
# =========================================================


# =========================================================
# LangChain 1.x Import (LCEL 기준 최소 구성)
# =========================================================

# Chat Prompt Template
from langchain_core.prompts import ChatPromptTemplate

# LCEL에서는 System / Human 클래스 직접 쓰는 경우 거의 없음
# System Message Template
# from langchain_core.prompts import SystemMessagePromptTemplate
# Human Message Template
# from langchain_core.prompts import HumanMessagePromptTemplate

# =========================================================
# Import 확인
# =========================================================

print("Prompt Import 성공")

Prompt Import 성공


In [ ]:
# =========================================================
# 역할:
# - System / Human Prompt 구성 (LCEL 실무 표준)
# - RAG용 프롬프트 설계 (SOURCES 포함)
# - ChatPromptTemplate 생성
# - LangChain 1.3.10 LCEL 방식 적용
# =========================================================

# =========================================================
# 실행 흐름
#
# system_template 정의
#        ↓
# human_template 정의
#        ↓
# ChatPromptTemplate.from_messages()
#        ↓
# LCEL Chain 입력 Prompt 완성
#        ↓
# ChatOpenAI 입력
#        ↓
# 답변 생성 (Markdown + SOURCES 포함)
# =========================================================


# =========================================================
# Import (LCEL 실무 표준)
# =========================================================

from langchain_core.prompts import ChatPromptTemplate

# =========================================================
# System Prompt (RAG 규칙 정의)
# =========================================================

system_template = """
Use the following pieces of context to answer the user's question shortly.

You MUST base your answer ONLY on the given context.

If the answer is not contained in the context, say "I don't know".

Always include SOURCES when relevant.

----------------
{context}

You MUST answer in Korean and in Markdown format.
"""

# =========================================================
# Human Prompt (사용자 질문)
# =========================================================

human_template = """
Question:
{question}
"""

# =========================================================
# Prompt 생성 (실무 LCEL 표준)
# =========================================================

prompt = ChatPromptTemplate.from_messages([
    ("system", system_template),
    ("human", human_template)
])

# =========================================================
# 확인 출력
# =========================================================

print("Prompt 생성 완료 (LCEL 실무 표준)")

Prompt 생성 완료 (LCEL 실무 표준)


문제) 다음 기존 코드 (보존)를 참고하여 코드를 완성하시오.

In [ ]:
# =========================================================
# 역할:
# - ChatOpenAI 생성
# - Retriever + Prompt + LLM LCEL 연결
# - RetrievalQAWithSourcesChain → LCEL 전환
# =========================================================

# =========================================================
# 실행 흐름
#
# 질문 입력
#     ↓
# Retriever (FAISS / Chroma)
#     ↓
# 문서 검색
#     ↓
# format_docs()
#     ↓
# Prompt 생성
#     ↓
# ChatOpenAI 호출
#     ↓
# StrOutputParser
#     ↓
# 최종 답변
# =========================================================

# =========================================================
# 기존 코드 (보존)
# =========================================================

'''
from langchain.chat_models import ChatOpenAI
from langchain.chains import RetrievalQAWithSourcesChain

chain_type_kwargs = {'prompt': prompt}

llm = ChatOpenAI(model_name='gpt-3.5-turbo', temperature=0)

chain = RetrievalQAWithSourcesChain.from_chain_type(
    llm=llm,
    chain_type='stuff',
    retriever=retriever,
    return_source_documents=True,
    chain_type_kwargs=chain_type_kwargs
)
'''

# =========================================================
# Import (LCEL 실무 표준)
# =========================================================

from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough


# =========================================================
# LLM 생성 (실무 표준)
# =========================================================

llm = ChatOpenAI(
    model= "gpt-4o-mini",
    temperature=0
)

# =========================================================
# 문서 포맷 함수
# =========================================================

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


# =========================================================
# LCEL RAG Chain (실무 표준)
# =========================================================

chain = (
    {
        "context" : retriever | format_docs,
        "question" : RunnablePassthrough()
    }
    | prompt | llm | StrOutputParser()
)


# =========================================================
# 테스트 실행
# =========================================================

response = chain.invoke("자율자동차의 지원은?")

print("\n====================")
print("답변")
print("====================")
print(response)


답변
자율주행차는 운전자의 간섭 없이 스스로 움직이며, 목적지까지 안전하게 이동할 수 있도록 설계되어 있습니다. 이는 사람의 실수를 줄여 교통사고를 감소시키는 데 기여합니다. 그러나 예측할 수 없는 상황이 발생할 수 있으며, 윤리적인 선택이 필요한 경우도 있습니다. 이러한 선택은 개인의 양심에 따라 결정되어야 하며, 인공지능(AI) 개발자와 사용자 윤리도 중요합니다. 

SOURCES: 제공된 텍스트.


In [ ]:
# =========================================================
# 역할:
# - 사용자 질문 정의
# - LCEL Chain 실행
# - RAG 기반 답변 생성
# =========================================================

# =========================================================
# 실행 흐름
#
# 질문 입력
#     ↓
# chain.invoke()
#     ↓
# Retriever
#     ↓
# 문서 검색
#     ↓
# format_docs()
#     ↓
# Prompt 생성
#     ↓
# LLM 호출
#     ↓
# 답변 출력
# =========================================================

# =========================================================
# 기존 코드 (보존)
# =========================================================

'''
query = '인공지능으로 인한 이슈는?'
result = chain(query)
print(result)
'''

# =========================================================
# 사용자 질문
# =========================================================

query = "인공지능으로 인한 이슈는?"

# =========================================================
# LCEL Chain 실행 (실무 표준)
# =========================================================

result = chain.invoke(query)

# =========================================================
# 결과 출력
# =========================================================

print("\n====================")
print("질문")
print("====================")
print(query)

print("\n====================")
print("답변")
print("====================")
print(result)


질문
인공지능으로 인한 이슈는?

답변
인공지능으로 인한 이슈는 다음과 같습니다:

1. **인간의 오용과 부주의**: 인공지능(AI) 자체는 두려움의 대상이 아니지만, 이를 개발하고 사용하는 인간에 의해 범죄에 사용될 수 있는 위험이 존재합니다.
   
2. **윤리와 가치**: 인공지능을 개발하는 과정에서 발생할 수 있는 실패와 자유의 침해에 대해 책임성 있게 행동해야 한다는 필요성이 강조됩니다.

3. **장기적인 문제**: 인공지능이 가져올 장기적인 문제에 대한 우려가 있으며, 미래 인공지능이 인류를 멸망하게 할 것이라는 강력한 가설은 삼가야 한다고 제시됩니다.

(SOURCES: 고인석, 2016)


In [ ]:
result

'인공지능으로 인한 이슈는 다음과 같습니다:\n\n1. **인간의 오용과 부주의**: 인공지능(AI) 자체는 두려움의 대상이 아니지만, 이를 개발하고 사용하는 인간에 의해 범죄에 사용될 수 있는 위험이 존재합니다.\n   \n2. **윤리와 가치**: 인공지능을 개발하는 과정에서 발생할 수 있는 실패와 자유의 침해에 대해 책임성 있게 행동해야 한다는 필요성이 강조됩니다.\n\n3. **장기적인 문제**: 인공지능이 가져올 장기적인 문제에 대한 우려가 있으며, 미래 인공지능이 인류를 멸망하게 할 것이라는 강력한 가설은 삼가야 한다고 제시됩니다.\n\n(SOURCES: 고인석, 2016)'

In [ ]:
# =========================================================
# 역할:
# - RAG 결과 출력
# - Retriever 기반 source_documents 확인
# - 문서 내용 / 파일 / 페이지 출력
# =========================================================

# =========================================================
# 실행 흐름
#
# query 입력
#     ↓
# chain.invoke(query) → 답변 생성
#     ↓
# retriever.invoke(query) → source 문서 검색
#     ↓
# doc.page_content / metadata 출력
# =========================================================

# =========================================================
# 사용자 질문
# =========================================================

query = "인공지능으로 인한 이슈는?"

# =========================================================
# 1. 답변 생성 (LCEL)
# =========================================================

result = chain.invoke(query)

print("\n====================")
print("답변")
print("====================")
print(result)

# =========================================================
# 2. Source 문서 직접 검색 (실무 방식)
# =========================================================

docs = retriever.invoke(query)

# =========================================================
# 3. Source 출력
# =========================================================

for doc in docs:

    print("\n====================")
    print("내용")
    print("====================")

    print(doc.page_content[:100].replace("\n", " "))

    print("\n파일 : " + str(doc.metadata.get("source", "N/A")))

    print("페이지 : " + str(doc.metadata.get("page", "N/A")))


답변
인공지능으로 인한 이슈는 다음과 같습니다:

1. **인간의 오용과 부주의**: 인공지능(AI) 자체는 두려움의 대상이 아니지만, 이를 개발하고 사용하는 인간에 의해 범죄에 사용될 수 있는 위험이 존재합니다.
   
2. **윤리와 가치**: 인공지능을 개발하는 과정에서 발생할 수 있는 실패와 자유의 침해에 대해 책임성 있게 행동해야 한다는 필요성이 강조됩니다.

3. **장기적인 문제**: 인공지능이 가져올 장기적인 문제에 대한 우려가 있으며, 미래 인공지능이 인류를 멸망하게 할 것이라는 강력한 가설은 삼가야 한다고 제시됩니다.

(SOURCES: 고인석, 2016)

내용
3) 인공지능(AI)의 특징인공지능(Artificial Intelligence)이란 인간처럼 사고하고 행동하는 기계나 시스템을 의미한다. 인공지능(AI)은 우리의 인간성에 대한 생

파일 : http://kocw-n.xcache.kinxcdn.com/data/keris/2023/keris0514/01.pdf
페이지 : 2

내용
이익을 위해 활용되어야 한다는 원칙을 포함하고 있다.Ÿ연구 이슈 : 아실로마 원칙이 추구하는 방향이 담겨있는데, 인간에게 유용하고 이로운 혜택을 주는 지능을 개발하는 것을 목표로 

파일 : http://kocw-n.xcache.kinxcdn.com/data/keris/2023/keris0514/01.pdf
페이지 : 7


In [ ]:
!pip install -qU gradio

In [ ]:
# =========================================================
# 역할:
# - LCEL 기반 PDF RAG 챗봇 + Gradio UI 연결
# - Retriever + LLM + Prompt 결과를 UI로 출력
# - Gradio 6.x messages format 호환 처리
# =========================================================

# =========================================================
# 실행 흐름
#
# 사용자 입력 (message)
#        ↓
# chain.invoke(message)
#        ↓
# LLM 답변 생성 (LCEL)
#        ↓
# retriever.invoke(message)
#        ↓
# 관련 문서 검색 (source_documents)
#        ↓
# bot_message 구성 (answer + sources)
#        ↓
# Gradio Chatbot 출력 (role/content format)
# =========================================================

# =========================================================
# 기존 코드 (보존 - tuple 방식 / legacy Gradio)
# =========================================================

'''
chat_history.append((message, bot_message))
'''

# =========================================================
# 변경 사항 (실무 적용 핵심)
# =========================================================

# 기존: tuple 기반 chat_history
#    ("user", "text")
#    ("bot", "text")

# 변경: Gradio 6.x messages format 적용
#    {"role": "user", "content": "..."}
#    {"role": "assistant", "content": "..."}

# 이유:
# - Gradio Chatbot이 messages mode 강제 환경
# - tuple 방식은 postprocess 단계에서 에러 발생

# =========================================================
# Import
# =========================================================

import gradio as gr

# =========================================================
# Chatbot 응답 함수
# =========================================================

def respond(message, chat_history):

    # -----------------------------------------------------
    # 1. LCEL Chain 실행 (답변 생성)
    # -----------------------------------------------------
    result = chain.invoke(message)

    # -----------------------------------------------------
    # 2. Retriever 실행 (source 문서 검색)
    # -----------------------------------------------------
    docs = retriever.invoke(message)

    # -----------------------------------------------------
    # 3. source 문자열 구성
    # -----------------------------------------------------
    sources = ""

    for i, doc in enumerate(docs):
        sources += f"\n\n[{i+1}] {doc.metadata.get('source', 'N/A')} ({doc.metadata.get('page','N/A')})"


    # -----------------------------------------------------
    # 4. 최종 메시지 구성
    # -----------------------------------------------------
    bot_message = result + sources

    # -----------------------------------------------------
    # 5. Gradio messages format 적용
    # -----------------------------------------------------
    chat_history.append({
        "role": "user",
        "content": message

    })

    chat_history.append({
        "role": "assistant",
        "content": bot_message

    })

    # -----------------------------------------------------
    # 6. 반환값 (입력 초기화 + 채팅 유지)
    # -----------------------------------------------------
    return "", chat_history

# =========================================================
# Gradio UI 구성
# =========================================================

with gr.Blocks() as demo:

    chatbot = gr.Chatbot(
        label="채팅창"
    )


    msg = gr.Textbox(label="입력")

    clear = gr.Button("초기화")

    # 메시지 입력 이벤트
    msg.submit(
        respond,
        inputs= [msg, chatbot],
        outputs= [msg, chatbot]

    )

    # 초기화 버튼
    clear.click(
        lambda:[],
        None,
        chatbot

    )

# =========================================================
# 실행
# =========================================================

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://8cbad1b3fb4f46d2be.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Colab + Gradio 에서 실행 가능한 pdf 기반 QA 챗봇
  - Gradio 기반 채팅 UI로 PDF 업로드 -> 질문 -> 대화처럼 쌓이는 LCEL RAG Q&A 챗봇


In [1]:
# =========================================================
# 1. 설치 (Colab 안정 버전)
# =========================================================
!pip install -q langchain langchain-openai langchain-community langchain-text-splitters faiss-cpu pypdf


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.9/121.9 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 24.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 33.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 558.5/558.5 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 2.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [4]:
# =========================================================
# 전체 시스템 구조
# =========================================================
# [FLOW DIAGRAM]
#
# PDF 업로드
#     ↓
# PyPDFLoader
#     ↓
# Text Split (chunking)
#     ↓
# Embedding (OpenAI)
#     ↓
# FAISS Vector DB 생성
#     ↓
# Retriever (Top-K 검색)
#     ↓
# LCEL Chain 구성
#     ↓
# Prompt + LLM (GPT)
#     ↓
# Response 생성
#     ↓
# Gradio Chat UI (Kakao Style)
# =========================================================

# =========================================================
# 2. IMPORT
# =========================================================
import os  # 환경변수
import gradio as gr  # UI 프레임워크

from google.colab import userdata  # Colab Secret Key

from langchain_openai import ChatOpenAI, OpenAIEmbeddings  # LLM + Embedding
from langchain_community.vectorstores import FAISS  # Vector DB
from langchain_community.document_loaders import PyPDFLoader  # PDF Loader
from langchain_text_splitters import RecursiveCharacterTextSplitter  # Text Splitter

from langchain_core.prompts import ChatPromptTemplate  # Prompt Template
from langchain_core.runnables import RunnablePassthrough, RunnableLambda  # LCEL Core


# =========================================================
# 3. API KEY SETUP (SAFE MODE)
# =========================================================
api_key = userdata.get("OPENAI_API_KEY")  # Colab secret에서 key 로드

if api_key is None:  # key 없으면 즉시 중단
    raise ValueError("OPENAI_API_KEY 없음")

os.environ["OPENAI_API_KEY"] = api_key  # LangChain에서 사용 가능하게 등록


# =========================================================
# 4. GLOBAL STATE (RAG 저장용)
# =========================================================
retriever = None  # Vector 검색기
rag_chain = None  # LCEL Chain


# =========================================================
# 5. PDF → VECTOR DB → LCEL RAG 생성
# =========================================================
def build_rag(pdf_file):

    global retriever, rag_chain  # 전역 상태 공유

    # -----------------------------
    # 5-1. PDF 로드
    # -----------------------------
    loader = PyPDFLoader(pdf_file.name)   # 업로드된 PDF 경로
    pages = loader.load()    # 페이지 단위 로드

    # -----------------------------
    # 5-2. 텍스트 분할 (Chunking)
    # -----------------------------
    splitter = RecursiveCharacterTextSplitter(
        chunk_size= 500,
        chunk_overlap = 50
    )


    docs = splitter.split_documents(pages)  # 문서 chunk 생성

    # -----------------------------
    # 5-3. Embedding 생성
    # -----------------------------
    embeddings =  OpenAIEmbeddings(
        model = "text-embedding-3-small"
    )  # embedding model

    # -----------------------------
    # 5-4. FAISS Vector DB 생성
    # -----------------------------
    vectorstore = FAISS.from_documents(docs, embeddings)    # vector DB 생성

    # -----------------------------
    # 5-5. Retriever 생성 (Top-K 검색)
    # -----------------------------
    retriever =  vectorstore.as_retriever(search_kwargs={"k":3})   # 상위 3개 검색

    # -----------------------------
    # 5-6. LLM 정의
    # -----------------------------
    llm = ChatOpenAI(
        model = "gpt-3.5-turbo",
        temperature= 0
    )


    # -----------------------------
    # 5-7. Prompt 정의
    # -----------------------------
    prompt = ChatPromptTemplate.from_template(
        """
        PDF 기반 AI 어시스턴트입니다.
        Context:
        {context}

        Question:
        {input}

        반드시 한국어로 답변하세요.
        """
    )


    # -----------------------------
    # 5-8. 문서 포맷 함수
    # -----------------------------
    def format_docs(docs):
        return  "\n\n".join(d.page_content for d in docs)  # 문서 합치기

    # -----------------------------
    # 5-9. LCEL Chain 구성
    # -----------------------------
    rag_chain = (
        {
            "context" : retriever | RunnableLambda(format_docs),
            "input" : RunnablePassthrough()
        }
        | prompt | llm
    )


    return "RAG READY"


# =========================================================
# 6. CHAT FUNCTION (Gradio Handler)
# =========================================================
def respond(message, chat_history):

    global rag_chain, retriever   # 전역 접근

    # -----------------------------
    # 6-1. RAG 준비 여부 체크
    # -----------------------------
    if rag_chain is None:
        chat_history.append({
            "role" : "assistant",
            "content" : "먼저 PDF를 업로드 하세요."
        })
        return "", chat_history


    # -----------------------------
    # 6-2. LCEL 실행
    # -----------------------------
    result =  rag_chain.invoke(message)  # LLM 결과

    # -----------------------------
    # 6-3. Retriever 실행
    # -----------------------------
    docs = retriever.invoke(message)   # 관련 문서 검색

    # -----------------------------
    # 6-4. Source 구성
    # -----------------------------
    sources = ""  # 초기화

    for i, doc in enumerate(docs):
        sources += f"\n\n[{i+1}] {doc.metadata.get('source', 'N/A')} \
        (page{doc.metadata.get('page', 'N/A')})"

    # -----------------------------
    # 6-5. 최종 메시지 생성
    # -----------------------------
    bot_message = result.content + sources    # 답변 + source

    # -----------------------------
    # 6-6. Chat history 업데이트
    # -----------------------------
    chat_history.append({

        "role" : "user",
        "content" : message
    })

    chat_history.append({

        "role" : "assistant",
        "content" : bot_message
    })

    return "", chat_history


# =========================================================
# 7. GRADIO UI
# =========================================================
with gr.Blocks() as demo:

    # -----------------------------
    # 7-1. Title
    # -----------------------------

    gr.Markdown("# LCEL PDF RAG Chatbot")

    # -----------------------------
    # 7-2. PDF Upload
    # -----------------------------

    file_input = gr.File(label="PDF 업로드")

    # -----------------------------
    # 7-3. Chat UI
    # -----------------------------
    chatbot = gr.Chatbot(height=500)


    # -----------------------------
    # 7-4. Input Box
    # -----------------------------
    msg = gr.Textbox(label="질문")


    # -----------------------------
    # 7-5. PDF Upload Event
    # -----------------------------
    file_input.upload(
        fn= build_rag,
        inputs= file_input,
        outputs= None

    )

    # -----------------------------
    # 7-6. Chat Event
    # -----------------------------
    msg.submit(
        fn=respond,
        inputs= [msg, chatbot],
        outputs= [msg, chatbot]

    )


# =========================================================
# 9. RUN
# =========================================================
demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://19c8e474c9a82038a8.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


문제) 다음 주어진 내용을 참고하여 카카오톡 대화창을 완성하시오. 단, css는 라이브코딩.

카톡 말풍선 적용 -> css 추가

In [ ]:
# =========================================================
# 1. INSTALL (Colab 안전 고정)
# =========================================================
!pip install -q langchain langchain-openai langchain-community langchain-text-splitters faiss-cpu pypdf gradio


In [ ]:
!pip install -qU gradio

In [ ]:
""""

PDF_AI_CHATBOT

main()

│
├── load_pdf()
│
├── split_documents()
│
├── create_embeddings()
│
├── create_vectorstore()
│
├── create_retriever()
│
├── create_rag_chain()
│
├── respond()
│
└── Gradio UI

"""

In [ ]:
"""

PDF 업로드
      │
      ▼
load_pdf()
      │
      ▼
split_documents()
      │
      ▼
create_embeddings()
      │
      ▼
create_vectorstore()
      │
      ▼
create_retriever()
      │
      ▼
create_rag_chain()
      │
      ▼
respond()
      │
      ▼
Gradio(ChatInterface)


"""

In [5]:
# ============================================================
# 프로젝트명
# ============================================================
#
# LangChain + LCEL 기반 PDF AI Chatbot
#
# 개발 환경
#
#   • Google Colab
#   • Python 3.12
#   • LangChain (Latest)
#   • OpenAI
#   • FAISS
#   • Gradio 6.x
#
# ============================================================



# ============================================================
# 전체 시스템 구조 (Flow Diagram)
# ============================================================
#
#                PDF 업로드
#                     │
#                     ▼
#                load_pdf()
#                     │
#                     ▼
#           split_documents()
#                     │
#                     ▼
#         create_embeddings()
#                     │
#                     ▼
#         create_vectorstore()
#                     │
#                     ▼
#          create_retriever()
#                     │
#                     ▼
#          create_rag_chain()
#                     │
#                     ▼
#               respond()
#                     │
#                     ▼
#        Gradio ChatInterface
#                     │
#                     ▼
#          KakaoTalk Style UI
#
# ============================================================



# ============================================================
# 1. Import Library
# ============================================================

# ------------------------------------------------------------
# 운영체제(OS) 관련 기능 사용
#
# • 환경 변수(Environment Variable) 설정
# • 파일 경로 처리
# ------------------------------------------------------------

import os

# ------------------------------------------------------------
# Gradio
#
# AI Chatbot UI 생성
# ------------------------------------------------------------

import gradio as gr

# ------------------------------------------------------------
# Google Colab Secret
#
# Colab Secret에 저장한
#
# OPENAI_API_KEY
#
# 를 안전하게 읽어오기 위해 사용
# ------------------------------------------------------------

from google.colab import userdata

# ------------------------------------------------------------
# OpenAI Chat Model
#
# GPT 모델 호출
# ------------------------------------------------------------

from langchain_openai import ChatOpenAI

# ------------------------------------------------------------
# OpenAI Embedding Model
#
# 문장을 숫자(Vector)로 변환
# ------------------------------------------------------------

from langchain_openai import OpenAIEmbeddings

# ------------------------------------------------------------
# PDF Loader
#
# PDF → LangChain Document
# ------------------------------------------------------------

from langchain_community.document_loaders import PyPDFLoader

# ------------------------------------------------------------
# FAISS
#
# Vector Database
# ------------------------------------------------------------

from langchain_community.vectorstores import FAISS


# ------------------------------------------------------------
# Text Splitter
#
# 긴 문서를
#
# 작은 Chunk로 분할
# ------------------------------------------------------------

from langchain_text_splitters import RecursiveCharacterTextSplitter

# ------------------------------------------------------------
# Prompt Template
#
# Prompt 생성
# ------------------------------------------------------------

from langchain_core.prompts import ChatPromptTemplate

# ------------------------------------------------------------
# LCEL
#
# Runnable 객체
# ------------------------------------------------------------

from langchain_core.runnables import RunnableLambda, RunnablePassthrough


# ============================================================
# 2. OpenAI API KEY 설정
# ============================================================

# ------------------------------------------------------------
# Google Colab Secret에서
#
# OPENAI_API_KEY
#
# 를 읽는다.
# ------------------------------------------------------------
api_key = userdata.get("OPENAI_API_KEY")


# ------------------------------------------------------------
# API Key가 존재하지 않으면
#
# 프로그램을 종료한다.
# ------------------------------------------------------------
if api_key is None:

    raise ValueError(
        "OPENAI_API_KEY가 존재하지 않습니다."
    )


# ------------------------------------------------------------
# LangChain이 사용할 수 있도록
#
# 환경 변수(Environment Variable)에 등록한다.
# ------------------------------------------------------------

os.environ["OPENAI_API_KEY"] = api_key


# ============================================================
# 3. Global Variable
# ============================================================

# ------------------------------------------------------------
# Retriever
#
# 역할
#
# 질문과 가장 유사한 문서를 검색한다.
# ------------------------------------------------------------

retriever = None

# ------------------------------------------------------------
# RAG Chain
#
# LCEL Pipeline을 저장한다.
# ------------------------------------------------------------


rag_chain = None

# ============================================================
# PART 1 완료
# ============================================================
#
# 현재까지 완료
#
#   ✔ Import
#   ✔ API KEY
#   ✔ Global Variable
#
# 다음 PART
#
#   load_pdf()
#
# ============================================================

In [17]:
# ============================================================
# 4. PDF 로드 함수
# ============================================================

def load_pdf(pdf):
    """
    ============================================================
    함수명
    ============================================================

    load_pdf()

    ============================================================
    함수 역할
    ============================================================

    사용자가 Gradio에서 업로드한 PDF 파일을 읽어
    LangChain Document 객체(List[Document])를 생성한다.

    이 함수는 "PDF를 읽는 역할만" 수행한다.

    즉,

    PDF
        ↓
    LangChain Document

    까지만 수행하며,

    Chunk 분할은 다음 PART에서 진행한다.

    ============================================================
    Parameter
    ============================================================

    pdf : gradio.UploadedFile

        Gradio File 컴포넌트에서 업로드한 PDF 파일

    ============================================================
    Return
    ============================================================

    list[Document]

        PDF를 페이지(Page) 단위로 읽은
        LangChain Document 객체 리스트

    ============================================================
    처리 과정
    ============================================================

    ① 업로드된 PDF의 실제 경로 확인

    ② PyPDFLoader 생성

    ③ PDF를 페이지 단위 Document 객체로 변환

    ④ Document 리스트 반환

    ============================================================
    """

    # --------------------------------------------------------
    # [Step 1]
    # 업로드된 PDF의 실제 저장 경로(Path)를 가져온다.
    #
    # Gradio는 업로드된 파일을 임시 폴더에 저장한다.
    #
    # 예)
    #
    # /tmp/gradio/xxxxxxxx/sample.pdf
    # --------------------------------------------------------
    pdf_path = pdf.name


    # --------------------------------------------------------
    # [Step 2]
    # PyPDFLoader 객체 생성
    #
    # 역할
    #
    # PDF 파일을 LangChain에서 사용할 수 있는
    # Document 객체로 변환하는 Loader이다.
    #
    # 아직 PDF를 읽지는 않는다.
    # --------------------------------------------------------
    loader = PyPDFLoader(pdf_path)


    # --------------------------------------------------------
    # [Step 3]
    # PDF 파일을 실제로 읽는다.
    #
    # 반환값
    #
    # List[Document]
    #
    # 각 Document 객체에는
    #
    # • page_content
    # • metadata
    #
    # 가 저장된다.
    # --------------------------------------------------------
    pages = loader.load()


    # --------------------------------------------------------
    # [Step 4]
    # 읽어온 페이지 개수를 출력한다.
    #
    # 강의용으로 매우 유용하다.
    # --------------------------------------------------------
    print("=" * 60)
    print("PDF 로드 완료")
    print(f"총 페이지 수 :  {len(pages)}")
    print("=" * 60)


    # --------------------------------------------------------
    # [Step 5]
    # 첫 번째 페이지 정보를 출력한다.
    #
    # Document 구조를 확인하기 위한 예제이다.
    #
    # 실제 서비스에서는 제거해도 된다.
    # --------------------------------------------------------
    if len(pages) > 0:

        print("\n첫 번째 Document 예시\n")

        print(pages[0])


    # --------------------------------------------------------
    # [Step 6]
    # Document 리스트를 반환한다.
    #
    # 다음 PART에서
    #
    # split_documents()
    #
    # 함수의 입력값으로 사용된다.
    # --------------------------------------------------------
    return pages

In [18]:
# ============================================================
# 5. Document Chunk 분할 함수
# ============================================================

def split_documents(pages):
    """
    ============================================================
    함수명
    ============================================================

    split_documents()

    ============================================================
    함수 역할
    ============================================================

    PDF를 읽어 생성한 Document 객체(List[Document])를

    작은 Chunk(Document) 단위로 분할한다.

    GPT는 긴 문서를 한 번에 입력받을 수 없기 때문에

    반드시 Chunking 과정이 필요하다.

    ============================================================
    Parameter
    ============================================================

    pages : list[Document]

        load_pdf() 함수가 반환한
        페이지 단위 Document 객체 리스트

    ============================================================
    Return
    ============================================================

    list[Document]

        Chunk 단위로 분할된
        Document 객체 리스트

    ============================================================
    처리 순서
    ============================================================

    ① RecursiveCharacterTextSplitter 생성

    ② Chunk Size 설정

    ③ Chunk Overlap 설정

    ④ Document 분할

    ⑤ Chunk 개수 출력

    ⑥ Chunk 리스트 반환

    ============================================================
    """

    # --------------------------------------------------------
    # [Step 1]
    # RecursiveCharacterTextSplitter 생성
    #
    # 역할
    #
    # 긴 문서를
    #
    # 작은 Document(Chunk)
    #
    # 로 분할한다.
    # --------------------------------------------------------
    splitter = RecursiveCharacterTextSplitter(

        # ----------------------------------------------------
        # 하나의 Chunk 최대 문자 수
        #
        # 예)
        #
        # 500
        #
        # 약 400~500자 정도가 하나의 Chunk가 된다.
        # ----------------------------------------------------

        chunk_size = 500,
        # ----------------------------------------------------
        # 이전 Chunk와 겹칠 문자 수
        #
        # Overlap을 주는 이유
        #
        # 문맥(Context)이 끊어지는 것을 방지하기 위해서이다.
        #
        # 예)
        #
        # Chunk1
        #
        # -------------------------
        # ABCDEFGHIJKLMNOP
        # -------------------------
        #
        # Chunk2
        #
        #            KLMNOPQRSTUV
        #
        # 마지막 일부를 공유한다.
        # ----------------------------------------------------
        chunk_overlap=50
    )

    # --------------------------------------------------------
    # [Step 2]
    # Document를 Chunk 단위로 분할한다.
    #
    # 반환값
    #
    # List[Document]
    #
    # Chunk마다 metadata는 유지된다.
    # --------------------------------------------------------
    docs = splitter.split_documents(pages)

    # --------------------------------------------------------
    # [Step 3]
    # 생성된 Chunk 개수를 출력한다.
    #
    # 강의 중
    #
    # Chunk Size
    #
    # 를 변경하며 비교하기 좋다.
    # --------------------------------------------------------
    print("=" * 60)
    print("Chunk 생성 완료")
    print(f"Chunk 개수 : {len(docs)}")
    print("=" * 60)

    # --------------------------------------------------------
    # [Step 4]
    # 첫 번째 Chunk를 출력한다.
    #
    # Document 구조를 확인하기 위한 예제이다.
    #
    # 실제 서비스에서는 제거 가능하다.
    # --------------------------------------------------------
    if len(docs) > 0:

        print("\n첫 번째 Chunk 예시\n")

        print(docs[0])

    # --------------------------------------------------------
    # [Step 5]
    # Chunk 리스트를 반환한다.
    #
    # 다음 PART에서
    #
    # Embedding을 생성한다.
    # --------------------------------------------------------
    return docs

In [19]:
"""

RAG의 핵심 흐름

Chunk Document
      │
      ▼
OpenAI Embedding
      │
      ▼

Vector(숫자 데이터)

"""

# ============================================================
# 6. Embedding Model 생성 함수
# ============================================================


def create_embeddings():
    """
    ============================================================
    함수명
    ============================================================

    create_embeddings()

    ============================================================
    함수 역할
    ============================================================

    텍스트 데이터를 숫자 벡터(Vector)로 변환하기 위한
    Embedding Model 객체를 생성한다.

    RAG 시스템에서는 문서를 그대로 검색하지 않는다.

    반드시

        Text

          ↓

        Embedding Vector

          ↓

        Similarity Search

    과정을 거친다.

    ============================================================
    Parameter
    ============================================================

    없음

    ------------------------------------------------------------

    API Key는 PART 1에서
    환경 변수로 등록되어 있기 때문에
    자동으로 사용된다.

    ============================================================
    Return
    ============================================================

    OpenAIEmbeddings

        Embedding Model 객체

    ============================================================
    처리 순서
    ============================================================

    ① OpenAI Embedding Model 선택

    ② Embedding 객체 생성

    ③ 생성된 객체 반환

    ============================================================
    """


    # --------------------------------------------------------
    # [Step 1]
    #
    # OpenAI Embedding 객체 생성
    #
    # 역할
    #
    # 문장을 숫자 Vector로 변환한다.
    #
    # 예)
    #
    # "LangChain은 AI Framework이다"
    #
    #          ↓
    #
    # [0.023, -0.154, 0.876 ...]
    #
    #
    # 이런 형태의 숫자 배열로 변환된다.
    #
    # --------------------------------------------------------

    embeddings = OpenAIEmbeddings(

        # ----------------------------------------------------
        # Embedding Model 선택
        #
        # text-embedding-3-small
        #
        # 특징
        #
        # ✔ 빠른 속도
        # ✔ 저렴한 비용
        # ✔ RAG에 적합
        #
        # ----------------------------------------------------

        model = "text-embedding-3-small"
    )

    # --------------------------------------------------------
    # [Step 2]
    #
    # 생성된 Embedding 객체 반환
    #
    # 이후 PART 5에서
    #
    # FAISS Vector DB 생성 시 사용한다.
    #
    # --------------------------------------------------------

    return embeddings


In [20]:
# ============================================================
# 7. FAISS Vector Store 생성 함수
# ============================================================


def create_vectorstore(docs, embeddings):
    """
    ============================================================
    함수명
    ============================================================

    create_vectorstore()

    ============================================================
    함수 역할
    ============================================================

    분할된 Document Chunk와 Embedding Model을 이용하여

    FAISS Vector Database를 생성한다.

    RAG 시스템에서 FAISS는

    "문서 저장소 + 빠른 유사도 검색 엔진"

    역할을 수행한다.

    ============================================================
    Parameter
    ============================================================

    docs : list[Document]

        split_documents() 함수에서 생성한
        Chunk Document 리스트


    embeddings : OpenAIEmbeddings

        create_embeddings() 함수에서 생성한
        Embedding Model 객체


    ============================================================
    Return
    ============================================================

    FAISS

        Vector Database 객체


    ============================================================
    처리 과정
    ============================================================

    ① Document Chunk 입력

    ② Embedding Model로 Vector 변환

    ③ FAISS Index 생성

    ④ Vector Store 반환

    ============================================================
    """


    # --------------------------------------------------------
    # [Step 1]
    #
    # FAISS Vector Store 생성
    #
    # 내부 처리 과정
    #
    #
    # Document
    #
    #    ↓
    #
    # Embedding Model
    #
    #    ↓
    #
    # Vector 생성
    #
    #    ↓
    #
    # FAISS Index 저장
    #
    #
    # --------------------------------------------------------

    vectorstore = FAISS.from_documents(

        # ----------------------------------------------------
        # 검색 대상이 되는 Document Chunk
        #
        # 예)
        #
        # [
        #   Document(chunk1),
        #   Document(chunk2),
        #   Document(chunk3)
        # ]
        #
        # ----------------------------------------------------

        docs,

        # ----------------------------------------------------
        # Text → Vector 변환 담당
        #
        # ----------------------------------------------------

        embeddings
    )


    # --------------------------------------------------------
    # [Step 2]
    #
    # 생성 완료 정보 출력
    #
    # 강의 및 테스트용
    # --------------------------------------------------------

    print("=" * 60)

    print("FAISS Vector Store 생성 완료")

    print(f"저장된 Chunk 개수 : {len(docs)}")

    print("=" * 60)



    # --------------------------------------------------------
    # [Step 3]
    #
    # 생성된 FAISS 객체 반환
    #
    # 다음 PART에서 Retriever 생성에 사용
    #
    # --------------------------------------------------------

    return vectorstore

In [21]:
# ============================================================
# 8. Retriever 생성 함수
# ============================================================


def create_retriever(vectorstore):
    """
    ============================================================
    함수명
    ============================================================

    create_retriever()

    ============================================================
    함수 역할
    ============================================================

    FAISS Vector Store를 이용하여
    Retriever 객체를 생성한다.

    Retriever는 사용자의 질문을 받아

        질문
          ↓
        Vector 변환
          ↓
        FAISS 검색
          ↓
        관련 Document 반환


    역할을 수행한다.

    즉 RAG 시스템에서

    "검색 담당 모듈"

    이다.

    ============================================================
    Parameter
    ============================================================

    vectorstore : FAISS

        create_vectorstore() 함수에서 생성한
        FAISS Vector Database


    ============================================================
    Return
    ============================================================

    Retriever

        질문과 관련된 Document를 검색하는 객체


    ============================================================
    처리 과정
    ============================================================

    ① FAISS Vector Store 입력

    ② Retriever 변환

    ③ 검색 옵션 설정

    ④ Retriever 반환

    ============================================================
    """


    # --------------------------------------------------------
    # [Step 1]
    #
    # Vector Store를 Retriever로 변환한다.
    #
    # FAISS는 저장소이고,
    # Retriever는 검색 인터페이스이다.
    #
    # 구조
    #
    # FAISS
    #
    #   ↓
    #
    # Retriever
    #
    #   ↓
    #
    # LangChain Chain 연결
    #
    # --------------------------------------------------------

    retriever = vectorstore.as_retriever(

        # ----------------------------------------------------
        # 검색 옵션 설정
        #
        # search_kwargs
        #
        # 검색 관련 파라미터 전달
        #
        # ----------------------------------------------------

          search_kwargs = {


            # ------------------------------------------------
            # k 의미
            #
            # Top-K 검색 개수
            #
            # 질문과 가장 유사한 문서를
            # 몇 개 가져올 것인지 결정
            #
            # 예)
            #
            # k=3
            #
            # 결과:
            #
            # Document 1
            # Document 2
            # Document 3
            #
            # ------------------------------------------------

            "k" : 3

          }

    )


    # --------------------------------------------------------
    # [Step 2]
    #
    # Retriever 생성 결과 출력
    #
    # --------------------------------------------------------

    print("=" * 60)

    print("Retriever 생성 완료")

    print("검색 문서 개수 : Top-3")

    print("=" * 60)



    # --------------------------------------------------------
    # [Step 3]
    #
    # Retriever 반환
    #
    # 다음 PART에서
    #
    # LCEL Chain 생성 시 사용
    #
    # --------------------------------------------------------

    return retriever

In [22]:
# ============================================================
# 9. LCEL 기반 RAG Chain 생성 함수
# ============================================================


def create_rag_chain(retriever):
    """
    ============================================================
    함수명
    ============================================================

    create_rag_chain()

    ============================================================
    함수 역할
    ============================================================

    Retriever와 LLM을 연결하여
    LangChain Expression Language(LCEL)
    기반 RAG Chain을 생성한다.

    최종 구조:

        사용자 질문

             ↓

        Retriever 검색

             ↓

        Context 생성

             ↓

        Prompt 조합

             ↓

        GPT 호출

             ↓

        답변 생성


    ============================================================
    Parameter
    ============================================================

    retriever :

        create_retriever() 함수에서 생성한
        문서 검색 객체


    ============================================================
    Return
    ============================================================

    Runnable Chain

        LCEL Pipeline 객체


    ============================================================
    처리 과정
    ============================================================

    ① ChatOpenAI 모델 생성

    ② Prompt Template 생성

    ③ Document Formatter 생성

    ④ LCEL Pipeline 구성

    ⑤ Chain 반환

    ============================================================
    """



    # ========================================================
    # 9-1. LLM 생성
    # ========================================================


    # --------------------------------------------------------
    # GPT 모델 객체 생성
    #
    # 역할
    #
    # 검색된 Context를 기반으로
    # 최종 답변을 생성한다.
    # --------------------------------------------------------

    llm = ChatOpenAI(

        # ----------------------------------------------------
        # 사용할 GPT 모델
        #
        # 비용과 성능 균형이 좋은 모델
        # ----------------------------------------------------
        model="gpt-3.5-turbo",


        # ----------------------------------------------------
        # temperature
        #
        # 답변의 창의성 조절
        #
        # 0
        #   → 일관적이고 정확한 답변
        #
        # 높을수록
        #   → 다양한 표현
        #
        # RAG에서는 보통 0 사용
        # ----------------------------------------------------
        temperature=0
    )



    # ========================================================
    # 9-2. Prompt Template 생성
    # ========================================================


    # --------------------------------------------------------
    # GPT에게 전달할 Prompt 생성
    #
    # {context}
    #
    # 검색된 문서 내용
    #
    # {input}
    #
    # 사용자 질문
    # --------------------------------------------------------

    prompt = ChatPromptTemplate.from_template(
        """

        당신은 PDF 문서 기반 AI Assistant입니다.


        아래 Context 정보를 이용해서
        질문에 답변하세요.


        Context:
        {context}


        Question:
        {input}


        반드시 한국어로 답변하세요.


        """
    )



    # ========================================================
    # 9-3. Document Formatter
    # ========================================================


    # --------------------------------------------------------
    # Retriever가 반환한 Document 리스트를
    # 하나의 문자열로 변환한다.
    #
    #
    # 입력:
    #
    # [
    # Document1,
    # Document2,
    # Document3
    # ]
    #
    #
    # 출력:
    #
    # "문서1 내용\n\n문서2 내용..."
    #
    # --------------------------------------------------------

    def format_docs(docs):

        # Document의 page_content만 추출

        return "\n\n".join(doc.page_content for doc in docs)



    # ========================================================
    # 9-4. LCEL Chain 구성
    # ========================================================


    # --------------------------------------------------------
    #
    # LCEL Pipeline
    #
    # 구성:
    #
    #
    # 질문
    #
    #   +
    #
    # Retriever Context
    #
    #   ↓
    #
    # Prompt
    #
    #   ↓
    #
    # GPT
    #
    #
    # --------------------------------------------------------

    rag_chain = (

        {


            # ------------------------------------------------
            # context 생성
            #
            # Retriever 실행
            #
            # 검색된 Document를
            # format_docs()로 변환
            # ------------------------------------------------

            "context": retriever | RunnableLambda(format_docs),




            # ------------------------------------------------
            # 사용자 질문 전달
            #
            # RunnablePassthrough는
            #
            # 입력값을 그대로 전달한다.
            #
            # ------------------------------------------------

            "input": RunnablePassthrough()


        }


        # Prompt 적용

        |

        prompt


        # GPT 실행

        |

        llm

    )



    # ========================================================
    # 9-5. Chain 반환
    # ========================================================


    print("=" * 60)

    print("LCEL RAG Chain 생성 완료")

    print("=" * 60)


    return rag_chain

In [28]:
# ============================================================
10. # RAG Chat Response
# ============================================================

def respond(
    message,
    chat_history
):

    global rag_chain
    global retriever


    # ==============================
    # PDF 없으면 일반 GPT
    # ==============================

    if rag_chain is None:

        llm = ChatOpenAI(
            model = "gpt-4o-mini",
            temperature=0
        )

        result = llm.invoke(message)

        answer = result.content


    # ==============================
    # PDF 있으면 RAG
    # ==============================

    else:

        result = rag_chain.invoke(
            message
        )

        answer = result.content


        docs = retriever.invoke(message)


        sources = "\n\n출처\n"


        for i, doc in enumerate(docs):

            sources += (
                f"{i+1}. "
                f"{doc.metadata.get('source','N/A')}"
                f" page:{doc.metadata.get('page','N/A')}\n"
            )


        answer += sources



    # ==============================
    # Gradio history
    # ==============================


    chat_history.append(
        {
            "role" : "user",
            "content" : message
        }
    )


    chat_history.append(
        {
            "role" : "assistant",
            "content" : answer
        }
    )


    return "", chat_history

In [29]:
from langchain_core import load
# ============================================================
# 11. KakaoTalk Style CSS
# ============================================================


# ------------------------------------------------------------
# CSS 역할
# ------------------------------------------------------------
#
# Gradio 기본 Chatbot 화면을
# 카카오톡 스타일의 말풍선 UI로 변경한다.
#
#
# 적용 내용
#
# ✔ 사용자 메시지 오른쪽 정렬
# ✔ AI 메시지 왼쪽 정렬
# ✔ 둥근 말풍선
# ✔ 메시지 여백
# ✔ 채팅창 배경
#
# ------------------------------------------------------------

# ============================================================
# PART 10
# Gradio UI
# ============================================================



kakao_css = """

.gradio-container {

    max-width:1400px !important;

    width:95% !important;

}


/* 전체 채팅 배경 */

.kakao-chat {

    background:#b2d9ff !important;

    border-radius:20px;

    padding:20px;

}



/* 입력창은 흰색 */

.kakao-input textarea {

    background:white !important;

    border-radius:25px !important;

    border:1px solid #ddd !important;

    padding:15px !important;

}



/* 사용자 메시지 노란색 */

.kakao-chat .message.user {

    background:#FEE500 !important;

    border-radius:18px !important;

}



/* AI 메시지 흰색 */

.kakao-chat .message.bot {

    background:white !important;

    border-radius:18px !important;

}


"""





# ============================================================
# PDF Process
# ============================================================


def process_pdf(pdf):


    global retriever

    global rag_chain



    pages = load_pdf(pdf)



    docs = split_documents (pages)



    embeddings = create_embeddings()



    vectorstore = create_vectorstore(
        docs, embeddings
    )



    retriever = create_retriever(vectorstore)


    rag_chain = create_rag_chain(retriever)


    return "PDF 분석 완료 - RAG 준비 완료"





# ============================================================
# Gradio
# ============================================================


with gr.Blocks() as demo:



    gr.Markdown(

        """
# PDF AI Assistant

LangChain + LCEL + FAISS RAG
"""

    )



    file_input = gr.File(

        label="PDF 업로드",

        file_types=[".pdf"]

    )



    status = gr.Textbox(

        label="상태",

        interactive=False

    )



    chatbot = gr.Chatbot(

        height=700,

        layout="bubble",

        render_markdown=False,

        elem_classes=[

            "kakao-chat"

        ]

    )



    msg = gr.Textbox(

        label=None,

        placeholder="메시지를 입력하세요",

        elem_classes=[
            "kakao-input"
        ]

    )



    file_input.upload(

        fn=process_pdf,

        inputs=file_input,

        outputs=status

    )



    msg.submit(

        fn=respond,

        inputs=[

            msg,

            chatbot

        ],

        outputs=[

            msg,

            chatbot

        ]

    )





demo.launch(

    share=True,

    css=kakao_css

)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://8cb6e8698aadb3e3f3.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


개선된 챗봇
  - 텔레그램 스타일
  - 속도향상
    1. st.rerun()없이 세션 상태로 처리 -> append()
    2. QA 체인 한번만 생성 -> 불필요한 재연산 제거

#다음 대화 시 이전 대화 사라지는 텔레그램

In [ ]:
!pip install -q streamlit langchain langchain-community langchain-openai chromadb pypdf pyngrok

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 36.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 58.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 38.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 25.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 22.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 56.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 54.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 41.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.9/178.9 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.9/61.9 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7

In [30]:
import time
import gradio as gr
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough


# =========================
# LLM + LCEL
# =========================
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

prompt = ChatPromptTemplate.from_template("""
질문:
{input}

한국어로 답변
""")

chain = (
    {"input": RunnablePassthrough()}
    | prompt
    | llm
)


# =========================
# CHAT (Gradio 6 규격 완벽 대응)
# =========================
def respond(message, chat_history):
    # 만약 기존 대화 기록이 없거나 튜플 형태 등으로 꼬여있다면 딕셔너리 리스트로 초기화
    if not chat_history or not isinstance(chat_history, list):
        chat_history = []

    # 1. AI 답변 생성
    result = chain.invoke(message)

    # 2. Gradio 6 규격에 맞게 딕셔너리 형태로 대화 추가
    chat_history.append({"role" : "user",
                         "content" : message})
    chat_history.append({"role" : "assistant",
                         "content" : result.content})

    # 3. 먼저 답변을 화면에 즉시 표시
    yield "", chat_history

    # 4. 정확히 2초 동안 대기
    time.sleep(2)

    # 5. 대화 기록을 완전히 비워서 화면에서 삭제 (Gradio 6용 빈 리스트 전달)
    chat_history = []
    yield "", chat_history


# =========================
# UI (요청하신 대로 아무 옵션도 넣지 않음)
# =========================
with gr.Blocks() as demo:

    gr.Markdown("# 텔레그램 CHATBOT")

    chatbot = gr.Chatbot()  # 옵션 없이 원본 유지

    msg = gr.Textbox()

    msg.submit(
        respond,
        [msg, chatbot],
        [msg, chatbot]
    )


demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://e216c3c0dabb94ba7c.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


# 일정시간 이후 대화 내용 사라지는 텔레그램 챗봇

In [32]:
!pip install -qU --force-reinstall chromadb langchain-community langchain-openai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.4/109.4 kB 6.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 1.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.0/104.0 kB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.7/41.7 kB 1.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 975.5 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 25.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 42.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 31.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.5/73.5 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7

In [6]:
import os
import time
import tempfile
import gradio as gr
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import CharacterTextSplitter

# ==========================================================
# 1. PDF 임베딩 및 LCEL 기반 RAG 체인 생성 함수
# ==========================================================
def process_pdf(file):
    if file is None:
        return "먼저 PDF 파일을 업로드해 주세요.", None

    try:
        tmp_path = file.name

        # PDF 로드 및 분할
        loader = PyPDFLoader(tmp_path)
        documents = loader.load()

        text_splitter = CharacterTextSplitter(
            chunk_size = 500,
            chunk_overlap = 50
        )
        docs = text_splitter.split_documents(documents)

        # 벡터 DB 생성 및 리트리버 설정
        embeddings = OpenAIEmbeddings()
        vectordb = Chroma.from_documents(docs, embeddings)
        retriever = vectordb.as_retriever(search_kwargs={"k":3}, search_type="mmr")

        # LLM 및 프롬프트 정의
        llm = ChatOpenAI(
            model="gpt-4o-mini",
            temperature=0
        )

        system_prompt = (
            "아래 제공된 문맥(context)만을 바탕으로 질문에 답하세요.\n"
            "상냥하고 친절하게 한국어로 답변해 주세요.\n\n"
            "Context:\n{context}\n\n"
            "질문: {input}"
        )
        prompt = ChatPromptTemplate.from_template(system_prompt)
        # 문서들을 하나의 문자열로 합쳐주는 내부 보조 함수
        def format_docs(docs):
            return "\n\n".join(doc.page_content for doc in docs)

        # 의존성 에러가 없는 최신 LCEL 파이프라인 체인 구축
        rag_chain = (
            {"context" : retriever | format_docs,
             "input" : RunnablePassthrough()}
            | prompt
            | llm
            | StrOutputParser()
        )


        return "PDF 임베딩 완료! 이제 질문을 입력하세요.", rag_chain
    except Exception as e:
        return f"오류 발생: {str(e)}", None

# ==========================================================
# 2. 챗봇 응답 및 10초 뒤 자동 삭제 함수
# ==========================================================
def respond(message, chat_history, rag_chain):
    if rag_chain is None:
        if chat_history is None:
            chat_history = []
        chat_history.append(
            {"role" : "assistant",
             "content" : "PDF 파일이 아직 업로드 되지 않았습니다."}
        )
        yield "", chat_history
        return

    if chat_history is None:
            chat_history = []

    # 사용자가 입력한 메시지 추가
    chat_history.append({"role" : "user",
                          "content" : message})
    yield "", chat_history

    # LCEL 체인은 입력 문자열을 그대로 던지면 스트링 응답을 바로 반환합니다.
    answer = rag_chain.invoke(message)

    # 챗봇 답변 추가

    chat_history.append({"role" : "assistant",
                         "content" : answer})
    yield "", chat_history

    # 10초 대기 후 메시지 폭파 (자동 삭제)

    time.sleep(10)

    chat_history = []
    yield "", chat_history

# ==========================================================
# 3. Gradio UI 구성
# ==========================================================
with gr.Blocks(css=".chatbot {height: 500px;}") as demo:
    gr.Markdown("# 텔레그램 스타일 PDF Q&A 챗봇")
    gr.Markdown("PDF를 업로드하고 자유롭게 질문하세요! (대화는 **10초 후 자동 삭제**됩니다.)")

    qa_state = gr.State(None)

    with gr.Row():
        with gr.Column(scale=3):
            file_input = gr.File(label=" PDF 업로드", file_types=[".pdf"])
            status_output = gr.Markdown("상태: PDF 업로드 대기 중...")

        with gr.Column(scale=7):
            chatbot = gr.Chatbot(elem_classes="chatbot")
            msg = gr.Textbox(label="질문을 입력하세요....")

    file_input.change(
        fn=process_pdf,
        inputs=file_input,
        outputs=[status_output, qa_state]
    )

    msg.submit(
        fn=respond,
        inputs=[msg, chatbot, qa_state],
        outputs=[msg, chatbot]
    )

demo.launch(share=True)

/tmp/ipykernel_47633/890950617.py:110: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: css. Please pass these parameters to launch() instead.
  with gr.Blocks(css=".chatbot {height: 500px;}") as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://203a1e76b61392ca67.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
